# 01 — OMI quotations: from raw data to an analytical panel

**Goal.** Build a transparent and reproducible view of Italian residential property quotations from the semiannual OMI releases.

The notebook follows a simple sequence: **inventory → load → validate → transform → define residential universe → describe the market**.

> **Important:** OMI publishes a minimum and maximum market quotation (€/m²) for a zone, property type and condition. The `Compr_mid` created below is the arithmetic midpoint of that interval. It is **not** an observed transaction price and it is **not** a statistical median.

**Source:** Agenzia delle Entrate — Osservatorio del Mercato Immobiliare (OMI).

## 1. Setup and source inventory

The project root is detected automatically, so the notebook can be run from the repository or from the `notebooks/` directory. File names provide the reference year and semester; those values are stored explicitly in the dataset.

In [ ]:
from pathlib import Path
import re

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 60)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')

def find_project_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / 'data' / 'raw' / 'quotations').is_dir():
            return candidate
    raise FileNotFoundError('Could not locate data/raw/quotations.')

PROJECT_ROOT = find_project_root()
RAW_DIR = PROJECT_ROOT / 'data' / 'raw' / 'quotations'
FILE_RE = re.compile(r'^omi_quotations_(\d{4})_(S[12])\.csv$', re.I)

catalogue = []
for path in sorted(RAW_DIR.glob('omi_quotations_*.csv')):
    match = FILE_RE.match(path.name)
    if not match:
        raise ValueError(f'Unexpected OMI filename: {path.name}')
    catalogue.append({'path': path, 'year': int(match.group(1)), 'semester': match.group(2).upper()})

catalogue = pd.DataFrame(catalogue).sort_values(['year', 'semester']).reset_index(drop=True)
if catalogue.empty:
    raise FileNotFoundError(f'No OMI quotation files found in {RAW_DIR}')

print(f'Project root: {PROJECT_ROOT}')
print(f'Files found: {len(catalogue):,}')
print(f'Coverage: {catalogue.year.min()}–{catalogue.year.max()}')
display(catalogue[['year', 'semester']])

## 2. Load and consolidate the releases

The raw files remain at zone/typology level. We deliberately postpone aggregation: first we want to know that all periods are present and that the key fields have the expected structure.

In [ ]:
frames = []
for row in catalogue.itertuples(index=False):
    part = pd.read_csv(row.path, sep=';', low_memory=False)
    part['reference_year'] = row.year
    part['semester'] = row.semester
    part['reference_date'] = pd.Timestamp(row.year, 6 if row.semester == 'S1' else 12, 30 if row.semester == 'S1' else 31)
    frames.append(part)

omi = pd.concat(frames, ignore_index=True)

print(f'Rows consolidated: {len(omi):,}')
print(f'Columns: {omi.shape[1]:,}')
display(omi[['reference_year','semester','Comune_ISTAT','Comune_descrizione','Zona','Descr_Tipologia','Compr_min','Compr_max']].head())

## 3. Validate before transforming

Three checks are deliberately separated from the analysis: **schema**, **time coverage** and **numeric quality**. This makes it clear whether a later result comes from the data or from a transformation error.

In [ ]:
required = {'Comune_ISTAT','Comune_cat','Comune_amm','Comune_descrizione','Zona','Descr_Tipologia','Compr_min','Compr_max'}
missing = sorted(required - set(omi.columns))
if missing:
    raise ValueError(f'Missing expected columns: {missing}')

periods = omi.groupby(['reference_year','semester']).size().rename('rows').reset_index()
expected = set(map(tuple, catalogue[['year','semester']].to_numpy()))
actual = set(map(tuple, periods[['reference_year','semester']].to_numpy()))
print('Missing periods:', sorted(expected - actual))

for col in ['Compr_min','Compr_max','Loc_min','Loc_max']:
    if col in omi.columns:
        omi[col] = pd.to_numeric(omi[col].astype('string').str.replace(',','.',regex=False), errors='coerce')

print(f'Missing Compr_min: {omi.Compr_min.isna().mean():.2%}')
print(f'Missing Compr_max: {omi.Compr_max.isna().mean():.2%}')

## 4. Create the analytical quotation variables

`Compr_mid` is the midpoint of the OMI range. `Compr_spread` and `Compr_spread_pct` describe the width of that range.

The OMI municipality identifier is stored as text because identifiers are keys, not quantities. The raw OMI representation contains an additional leading digit; the canonical six-digit municipal code is extracted for future integration.

In [ ]:
omi['Compr_mid'] = omi[['Compr_min','Compr_max']].mean(axis=1)
 omi_spread = omi['Compr_max'] - omi['Compr_min']
omi['Compr_spread'] = omi_spread
omi['Compr_spread_pct'] = omi_spread.div(omi['Compr_mid'].replace(0,np.nan)).mul(100)

raw_istat = pd.to_numeric(omi['Comune_ISTAT'], errors='coerce').astype('Int64').astype('string')
omi['municipality_code_omi'] = raw_istat
omi['municipality_code'] = raw_istat.str[-6:]

display(omi[['Compr_min','Compr_max','Compr_mid','Compr_spread','Compr_spread_pct']].describe().T)

## 5. Define the residential universe

The downstream project is about residential real estate. We therefore keep typologies whose description contains `abitazion` or `villa`. The rule is intentionally visible rather than hidden in a helper function, so a reader can audit or change it.

The result remains a **zone-level** dataset: one municipality can contribute many observations in the same semester because it can contain multiple OMI zones, typologies and conditions.

In [ ]:
residential_mask = omi['Descr_Tipologia'].astype('string').str.contains('abitazion|villa', case=False, na=False)
residential = omi.loc[residential_mask & omi['Compr_mid'].notna()].copy()

print(f'Residential observations: {len(residential):,}')
print(f'Unique municipalities: {residential.municipality_code.nunique():,}')
print(f'Unique semesters: {residential.reference_date.nunique():,}')
display(residential['Descr_Tipologia'].value_counts().rename('observations').to_frame())

## 6. National and regional descriptive analysis

The national series uses the **median of zone-level `Compr_mid` values**. This is a distributional statistic of OMI quotations; it is not weighted by transactions or population.

For the regional snapshot we also report the number of observations and municipalities, because coverage is part of the interpretation.

In [ ]:
national_trend = (residential.groupby('reference_date',as_index=False).agg(median_omi_m2=('Compr_mid','median'),mean_omi_m2=('Compr_mid','mean'),observations=('Compr_mid','count'),municipalities=('municipality_code','nunique')).sort_values('reference_date'))
display(national_trend.tail(10))

fig, ax = plt.subplots(figsize=(11,5))
ax.plot(national_trend['reference_date'], national_trend['median_omi_m2'])
ax.set(title='Residential OMI quotation midpoint — national median by semester',xlabel='Reference semester',ylabel='€/m²')
ax.grid(alpha=.25)
plt.show()

latest_date = residential['reference_date'].max()
latest_regional = (residential.loc[residential.reference_date.eq(latest_date)].groupby('Regione',as_index=False).agg(median_omi_m2=('Compr_mid','median'),observations=('Compr_mid','count'),municipalities=('municipality_code','nunique')).sort_values('median_omi_m2',ascending=False))
display(latest_regional)

## 7. What this notebook establishes

1. **Reproducible input:** all available OMI semester files are discovered and loaded.
2. **Auditable transformations:** numeric conversion, midpoint and spread calculations are explicit.
3. **Clear market definition:** residential typologies are separated from non-residential ones.
4. **Correct interpretation:** OMI quotations are ranges by zone/typology/condition, not observed sale prices.

The next notebook uses a different source concept — **NTN (Numero di Transazioni Normalizzate)** — and therefore keeps transaction volumes separate until the controlled integration stage.